In [2]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspace\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


In [3]:
# PDF에서 표, 텍스트, 이미지 같은 요소를 추출하기 위해 사용되는 기능
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf

def extract_pdf_elements(path, fname): # PDF에서 elements를 추출하는 함수
    """
    PDF 파일에서 이미지, 테이블, 텍스트 블록을 추출합니다.
    path: 파일 경로 (추출된 이미지 파일이 저장될 위치)
    fname: 파일 이름
    """
    return partition_pdf( # unstructured가 주는 함수
        filename=path + fname,
        strategy="hi_res",              # 레이아웃 분석 모델로 정밀 추출
        extract_images_in_pdf=True,     # PDF에서 이미지를 추출
        infer_table_structure=True,     # 테이블 구조를 추론
        chunking_strategy="by_title",   # 타이틀을 기준으로 텍스트를 블록으로 분할
        max_characters=4000,            # 최대 4000자로 텍스트 블록을 제한
        new_after_n_chars=3000,         # 3000자 이후에 새로운 블록 생성
        combine_text_under_n_chars=2000, # 2000자 이하의 텍스트는 결합
        extract_image_block_output_dir=path+"/project",     # 이미지가 저장될 경로 설정
    )

In [4]:
def categorize_elements(raw_pdf_elements): # elements를 분류
    """
    PDF에서 추출한 요소들을 테이블과 텍스트로 분류합니다.
    raw_pdf_elements: unstructured.documents.elements 리스트
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))     # 테이블 요소를 저장
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))      # 텍스트 요소를 저장
    return texts, tables


In [6]:
import os

# 현재 작업 디렉터리 경로를 반환합니다.
current_directory = os.getcwd()

# fname: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "assignment.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "data/" )

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

# 실행하면 2~3분 걸린다. 멈춘게 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)

No languages specified, defaulting to English.


현재 스크립트의 위치: c:\workspace\multimodal-rag\notebooks
pdf 위치: c:\workspace\multimodal-rag\data/


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2281.90it/s]
